# Sentiment Analysis Project

**Goal:** Measure public opinion by classifying text into positive or negative sentiment using computational linguistics, NLP, and machine learning.

This notebook provides a complete, runnable pipeline from data loading to advanced inference using state-of-the-art Large Language Models.

## 1. Setup & Imports
Run the following cell to install the required libraries and import necessary modules. This ensures the notebook runs perfectly from top to bottom.

In [ ]:
!pip install -q pandas scikit-learn transformers nltk seaborn matplotlib

import pandas as pd
import numpy as np
import nltk
import re
import matplotlib.pyplot as plt
import seaborn as sns

# Machine Learning Imports
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Deep Learning / Hugging Face Imports
from transformers import pipeline

# Download necessary NLTK datasets for our corpora and tokenization
nltk.download('movie_reviews', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt', quiet=True)
from nltk.corpus import movie_reviews
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import warnings
warnings.filterwarnings('ignore')

## 2. Data Loading & Preparation
We will use the **IMDB Movie Reviews** dataset built into `nltk`. It contains 2,000 movie reviews (1,000 positive, 1,000 negative). We will load it directly into a Pandas DataFrame for easy manipulation, making it 100% reproducible without manual downloads.

In [ ]:
print("Loading dataset...")
documents = []
for category in movie_reviews.categories():
    for fileid in movie_reviews.fileids(category):
        # Extract text for the review
        text = movie_reviews.raw(fileid)
        documents.append((text, category))

# Create DataFrame
df = pd.DataFrame(documents, columns=['text', 'label'])

# Map labels to binary values for our ML Model: 'pos' -> 1, 'neg' -> 0
df['target'] = df['label'].map({'pos': 1, 'neg': 0})

print(f"Dataset loaded successfully! Shape: {df.shape}")
display(df.head())

## 3. Text Preprocessing
Raw text data is messy. We'll use `nltk` to clean the text by:
- Converting to lowercase.
- Removing punctuation and numbers.
- Tokenizing (splitting text into individual words).
- Removing stopwords (common words like 'the', 'and', 'is' that don't hold much sentiment value).

In [ ]:
stop_words = set(stopwords.words('english'))

def preprocess_text(text):
    # 1. Lowercase
    text = text.lower()
    # 2. Remove non-alphabetic characters using regex
    text = re.sub(r'[^a-z\s]', '', text)
    # 3. Tokenize
    tokens = word_tokenize(text)
    # 4. Remove stopwords
    tokens = [word for word in tokens if word not in stop_words]
    # 5. Rejoin into a single string
    return ' '.join(tokens)

print("Preprocessing text (this may take a moment)...")
df['clean_text'] = df['text'].apply(preprocess_text)
print("Preprocessing complete!")
display(df[['text', 'clean_text', 'label']].head())

## 4. Baseline Model (Machine Learning)
We will extract features using **TF-IDF (Term Frequency-Inverse Document Frequency)** and train a baseline **Logistic Regression** model.

In [ ]:
# Split the data into Training and Testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], 
    df['target'], 
    test_size=0.2, 
    random_state=42, 
    stratify=df['target']
)

# Initialize TF-IDF Vectorizer to convert text to numerical vectors
vectorizer = TfidfVectorizer(max_features=5000)

# Fit on training data and transform both train and test data
X_train_vec = vectorizer.fit_transform(X_train)
X_test_vec = vectorizer.transform(X_test)

print(f"TF-IDF Vocabulary size: {len(vectorizer.vocabulary_)}")

# Train the Logistic Regression Model
print("Training Logistic Regression model...")
lr_model = LogisticRegression(random_state=42)
lr_model.fit(X_train_vec, y_train)
print("Model training complete!")

## 5. Evaluation & Metrics
Let's evaluate our baseline model using standard classification metrics: **Accuracy**, **F1-Score**, and a **Confusion Matrix**.

In [ ]:
# Make predictions on the test set
y_pred = lr_model.predict(X_test_vec)

# Calculate metrics
acc = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)

print(f"Baseline Model (Logistic Regression) - Accuracy: {acc:.4f}")
print(f"Baseline Model (Logistic Regression) - F1-Score: {f1:.4f}")

# Generate Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

# Plot Confusion Matrix using Seaborn
plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Negative (0)', 'Positive (1)'], 
            yticklabels=['Negative (0)', 'Positive (1)'])
plt.title('Confusion Matrix - Baseline Model')
plt.xlabel('Predicted Label')
plt.ylabel('True Label')
plt.show()

## 6. Advanced Model (Deep Learning / LLM)
Instead of training a model from scratch with TF-IDF, modern NLP relies on powerful pre-trained transformer models. 

Here, we demonstrate **Zero-Shot Inference** using Hugging Face's `transformers` library. We will load a `distilbert` pipeline pre-trained specifically for sentiment analysis.

In [ ]:
print("Downloading and initializing the pre-trained Hugging Face model...")
# Initialize a pre-trained sentiment analysis pipeline
# This uses DistilBERT fine-tuned on the SST-2 sentiment dataset
sentiment_pipeline = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")

print("\n--- Hugging Face Transformers Inference Demonstration ---\n")

# Test it on a few randomly selected samples from our test set
sample_indices = y_test.sample(5, random_state=42).index

for idx in sample_indices:
    original_text = df.loc[idx, 'text']
    true_label = df.loc[idx, 'label']
    
    # Truncate text to 512 characters to avoid token limits for this demonstration
    short_text = original_text[:512]
    
    # Get prediction from the Hugging Face pipeline
    prediction = sentiment_pipeline(short_text)[0]
    
    print(f"Review Snippet: \"{short_text[:100]}...\"")
    print(f"True Label: {true_label.upper()}")
    print(f"HF Prediction: {prediction['label']} (Confidence Score: {prediction['score']:.4f})\n")
    print("-" * 60)